# Churn SaaS — pipeline de données

Estimer la probabilité de résiliation d'un client à l'échéance, à partir de
`docs/churn_saas_complet.csv` (5035 lignes, 28 % de churn).

**Prérequis** : `docker compose up -d` — PostgreSQL sur le port 5432, pgAdmin sur
http://localhost:5050.

## 🥉🥈🥇 Ingestion complète

Lance les trois couches à la suite : `docs/*.csv` → 🥉 bronze → 🥈 silver →
🥇 gold. Chaque couche est intégralement rechargée, l'exécution est donc
rejouable sans créer de doublons.

In [ ]:
from ml_churn.ingestion.scripts.ingest_all import ingest_all

lignes_ingestion = ingest_all()

## 1. Exploration des données brutes

Deux types de graphiques construits directement sur le CSV, avant toute
ingestion. Les valeurs sont normalisées à la volée — le CSV mélange les casses
(`TPE`, `tpe`, `" TPE "`) et les unités (`20.0%`, `3.1 h`, `280.62 €`).

**Histogrammes** — répartition des clients par secteur, pays, taille
d'entreprise et plan, puis distribution des variables d'usage (ancienneté,
CSAT, délai de réponse du support, revenu mensuel…).

**Boîtes à moustaches** — complémentaires : là où l'histogramme montre la forme
de la distribution, le boxplot isole individuellement chaque point au-delà de
1,5 × IQR, même s'il n'y en a qu'un sur 5000. Les variables très asymétriques
sont tracées en échelle logarithmique ; en linéaire, le critère IQR classe
toute la traîne en valeurs extrêmes et écrase la boîte.

Les PNG sont exportés dans `src/visualization/graphs/<type>/<préfixe>_<donnée>.png`.

In [ ]:
import sys

sys.path.insert(0, "src")  # src/visualization n'est pas un package installe
from visualization.scripts.plot_all import plot_all

graphiques = plot_all()

## 🥉 2. Ingestion — couche bronze

Les trois CSV de `docs/` sont copiés **tels quels** dans le schéma `bronze` :
toutes les colonnes métier en `TEXT`, aucune conversion ni nettoyage. Chaque ligne
conserve son origine (`_source_file`, `_source_line`, `_ingested_at`).

Les tables sont vidées puis rechargées : relancer la cellule ne crée pas de
doublons. Le log compare les lignes lues dans le CSV à celles réellement insérées
en base, et signale tout écart.

In [ ]:
from ml_churn.ingestion.scripts.ingest_bronze import ingest_bronze

lignes_bronze = ingest_bronze()

## 🥈 3. Ingestion — couche silver

`bronze.churn_saas_complet_bronze` → `silver.churn_saas_silver`, en onze actions
successives :

1. **Déduplication** sur `client_id`
2. **Standardisation** de huit colonnes : dates ramenées au format `AAAA-MM-JJ`,
   pays en codes ISO, plans en `PRO`/`BUS`/`STR`/`ENT`, casse et espaces harmonisés
3. **Règles métier** : toute ligne hors bornes est supprimée
4. **Typage** : `date`, `integer` et `numeric` selon la colonne

In [ ]:
from ml_churn.ingestion.scripts.ingest_silver import ingest_silver

lignes_silver = ingest_silver()

## 🥇 4. Ingestion — couche gold

`silver` → `gold`, pour les deux tables : `catalogue_gold` et
`churn_saas_gold`. Les tables reprennent **la structure de silver à
l'identique** (mêmes colonnes, mêmes types) — les définitions sont partagées
via les mixins de `models/colonnes.py`, il n'y a donc pas deux schémas à
maintenir.

Gold est la couche stable sur laquelle s'appuient l'analyse et la
modélisation, sans dépendre des retraitements successifs de silver.

In [ ]:
from ml_churn.ingestion.scripts.ingest_gold import ingest_gold

lignes_gold = ingest_gold()

## 5. Déséquilibre de la cible

Effectifs de chaque valeur de `churn` et taux de résiliation, lus dans
`gold.churn_saas_gold`. Le code est dans `src/ml_churn/training/`.

Ce déséquilibre conditionne le choix des métriques : un modèle qui prédirait
« personne ne résilie » aurait déjà une exactitude égale à la part de clients
actifs.

In [ ]:
from ml_churn.training.classification.analyze_target import analyze_target

repartition_cible = analyze_target()

## 6. Classification — baseline (régression logistique)

**Recherche du seuil.** Les 11 seuils de 0,0 à 1,0 (pas de 0,1) testés sur le modele pour évaluer lequel correspond le plus à notre besoin

**Entraînement.** Le modèle est ensuite évalué au seuil retenu. Remplace `SEUIL`
par une valeur fixe pour comparer les arbitrages : un seuil bas détecte plus de
churners au prix de fausses alertes, un seuil haut fait l'inverse.

Le seuil retenu est de 0.5 (recall = 0.8 / FPR = 0.19)

Pour explorer les runs et la matrice de confusion enregistrée :

```bash
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db
```

In [1]:
from ml_churn.training.classification.baseline.classification_baseline_training import (
    train_classification_baseline,
)
from ml_churn.training.classification.baseline.classification_baseline_tuning import (
    tune_baseline_threshold,
)

resultat_tuning = tune_baseline_threshold()

SEUIL = resultat_tuning.seuil  # ou une valeur fixee a la main

resultat_baseline = train_classification_baseline(seuil=SEUIL)

/Users/avallet/machine-learning/ml-churn/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[RECHERCHE] seuil, objectif 'f1', 11 seuils de 0 a 1 (pas 0.1)
  train 3000 / validation 1000 / test 1000


2026/09/21 18:59:35 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/avallet/machine-learning/ml-churn
/Users/avallet/machine-learning/ml-churn/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/Users/avallet/machine-learning/ml-churn/.venv/lib/python3.13/site


[SEUIL RETENU] 0.5000  (f1 = 0.6855 en validation)

[PERFORMANCE] sur le jeu de test, au seuil retenu
  accuracy              0.8080
  recall                0.8000
  false_positive_rate   0.1889
  precision             0.6222

[TOUS LES SEUILS TESTES]
    seuil   score   recall  precision     FPR
   0.0000  0.4375   1.0000     0.2800  1.0000
   0.1000  0.5560   0.9750     0.3889  0.5958
   0.2000  0.6119   0.9571     0.4497  0.4556
   0.3000  0.6325   0.8821     0.4930  0.3528
   0.4000  0.6630   0.8536     0.5420  0.2806
   0.5000  0.6855   0.7786     0.6124  0.1917
   0.6000  0.6828   0.7071     0.6600  0.1417
   0.7000  0.6488   0.6036     0.7012  0.1000
   0.8000  0.5951   0.4750     0.7964  0.0472
   0.9000  0.4845   0.3357     0.8704  0.0194
   1.0000  0.0000   0.0000     0.0000  0.0000

  runs enregistres dans /Users/avallet/machine-learning/ml-churn/mlflow.db (uv run mlflow ui)
[SEUIL] 0.50
[DONNEES] : 5000 clients, 64 features
  exclues : client_id, code_datacenter, couleur_t